In [151]:
import os
print(f"Компютер має {os.cpu_count()} ядера")

Компютер має 4 ядера


# 1. Write a version of curried function c_5 using lambda syntax.

In [154]:
c_5 = lambda a: lambda b: lambda c: lambda d: lambda e: a + b + c + d + e
print(c_5(1)(2)(3)(4)(5))

15


# 2. Write a program calculating a factorial using parallelisation and map-reduce.

In [155]:
from functools import reduce
from pathos.multiprocessing import ProcessPool

n = 5
numbers_to_multiply = list(range(1, n + 1))
# Власна реалізація reduce (рекурсивна)
def reduce(f, id, s):  
    l = len(s)  
    if l == 0:  
        return id  
    elif l == 1:  
        return s[0]  
    else:  
        return f(reduce(f, id, s[1:]), s[0])
multiply = lambda x, y: x * y  
# MAP-фаза (паралельно)
def parallel_map(lst):
    with ProcessPool() as pool:
        return pool.map(lambda x: x, lst)
# Паралельне "відображення" чисел
mapped_numbers = parallel_map(numbers_to_multiply)
# REDUCE-фаза (послідовно, як у твоєму коді)
res = reduce(multiply, 1, mapped_numbers)
print(f"{n}! =", res)


5! = 120


# 3. Write a parallelised version of reduce (hint: use chunking).

In [156]:
import math
def parallel_reduce(f, id, data, num_processes=4):
    if not data:
        return id
    def chunkify(lst, num_chunks):
        avg = math.ceil(len(lst) / num_chunks)
        return [lst[i*avg:(i+1)*avg] for i in range(num_chunks)]
    chunks = chunkify(data, num_processes)
    
    def local_reduce(sublist):
        res = id
        for x in sublist:
            res = f(res, x)
        return res

    with ProcessPool(nodes=num_processes) as pool:
        partial_results = pool.map(local_reduce, chunks)
    result = id
    for r in partial_results:
        result = f(result, r)
    return result
    
res = parallel_reduce(multiply, 1, mapped_numbers)
print(f"{n}! =", res)

5! = 120


# 4. Compare performance of Numpy vectorization vs parallel map on a really large array of integers.


In [157]:
import numpy as np
import time

n = 100_000
number_list = list(range(1, n + 1))

start_time = time.time()
np_array = np.array(number_list)
vectorized_fn = np.vectorize(lambda x: x+x)
result = vectorized_fn(np_array)
print(result)
end_time = time.time()
print(f"NumPy vectorization час: {end_time - start_time:.5f} сек")

start_time = time.time()
with ProcessPool() as pool:
    result_parallel = pool.map(lambda x: x+x, number_list)
end_time = time.time()
print(f"Parallel map час: {end_time - start_time:.5f} сек")




[     2      4      6 ... 199996 199998 200000]
NumPy vectorization час: 0.04821 сек
Parallel map час: 0.59508 сек


# 5. Using computations from previous labs, apply chunking approach to df.apply(...) using parallel map.

In [1]:
import pandas as pd
import numpy as np
hc = pd.read_csv("data/heart_2020_cleaned.csv")

In [2]:
from pathos.multiprocessing import ProcessPool
# --- Функція для обчислення середнього ---
def compute_mean(series):
    return series.mean()

# --- Розбивка DataFrame на chunks ---
def chunkify_hc(df, num_chunks):
    chunk_size = int(np.ceil(len(df) / num_chunks))  # перевір, щоб int був функцією
    return [df.iloc[i*chunk_size:(i+1)*chunk_size] for i in range(num_chunks)]

# --- Паралельне застосування ---
def parallel_apply_mean(df_chunks, column):
    with ProcessPool(nodes=len(df_chunks)) as pool:
        # для кожного chunk беремо серію та обчислюємо середнє
        results = pool.map(lambda df_chunk: df_chunk[column].mean(), df_chunks)
    # усереднюємо результати chunk-ів
    overall_mean = sum(results) / len(results)
    return overall_mean

# --- Виконання ---
num_processes = 4
hc_chunks = chunkify_hc(hc, num_processes)

bmi_mean = parallel_apply_mean(hc_chunks, 'BMI')
print(f"Середнє значення стовпця BMI: {bmi_mean:.2f}")


Середнє значення стовпця BMI: 28.33
